## Импорт библиотек и данных

In [2]:
import joblib
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append('../')
from src import *

In [3]:
X_train = joblib.load('../data/01/X_train_raw.pkl')
y_train = joblib.load('../data/01/y_train.pkl')

X_valid = joblib.load('../data/01/X_valid_raw.pkl')
y_valid = joblib.load('../data/01/y_valid.pkl')

X_test = joblib.load('../data/01/X_test_raw.pkl')
y_test = joblib.load('../data/01/y_test.pkl')

result = joblib.load('../data/04/models_gini_not_fe.pkl')

## Добавление новых признаков

In [4]:
def add_all_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_mmr_ratios(df)
    df = add_usage_features(df)
    return df


In [5]:
X_train = add_all_features(X_train)
X_valid = add_all_features(X_valid)
X_test  = add_all_features(X_test)

In [6]:
for col in ["Make", "Auction", "Transmission", "Color"]:
    X_train, X_valid, X_test = add_frequency_features(X_train, X_valid, X_test, col)

## Переобучаем модели

In [7]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_valid = pd.DataFrame(
    scaler.transform(X_valid),
    columns=X_valid.columns,
    index=X_valid.index
)

X_test = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [8]:
# Logistic Regression
sk_lr = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=5000
)
sk_lr.fit(X_train, y_train)
lr_pred = sk_lr.predict_proba(X_valid)[:, 1]
print("Sklearn Logistic Regression Gini:", gini_score(y_valid, lr_pred))

Sklearn Logistic Regression Gini: 0.4636741599156169


In [9]:
# KNN (можно использовать подвыборку для скорости)
sk_knn = KNeighborsClassifier(n_neighbors=15)
sk_knn.fit(X_train[:2000], y_train[:2000])
knn_pred = sk_knn.predict_proba(X_valid)[:, 1]
print("Sklearn KNN Gini:", gini_score(y_valid, knn_pred))

Sklearn KNN Gini: 0.3330278103209743


In [10]:
# GaussianNB
sk_gnb = GaussianNB()
sk_gnb.fit(X_train, y_train)
gnb_pred = sk_gnb.predict_proba(X_valid)[:, 1]
print("Sklearn GaussianNB Gini:", gini_score(y_valid, gnb_pred))

Sklearn GaussianNB Gini: 0.4483543759059645


## Результат

In [11]:
result['gini(FE)'] = [gini_score(y_valid, lr_pred), gini_score(y_valid, knn_pred), gini_score(y_valid, gnb_pred)]
result['delta'] = result['gini(FE)'] - result['gini(not FE)']

In [12]:
result

,Algorithm,gini(not FE),gini(FE),delta
0,Logistic Regression,0.462344,0.463674,0.001330
1,KNN,0.326679,0.333028,0.006349
2,GaussianNB,0.447247,0.448354,0.001107


## Созранение результатов

In [13]:
joblib.dump(X_train, '../data/05/X_train_up.pkl')
joblib.dump(X_valid, '../data/05/X_valid_up.pkl')
joblib.dump(X_test, '../data/05/X_test_up.pkl')
joblib.dump(result, '../data/05/models_gini_up.pkl')

['../data/05/models_gini_up.pkl']